In [ ]:
import pandas as pd

matrix_path = "GSE157103_series_matrix.txt"
tpm_path = "GSE157103_genes.tpm.tsv"

# 1. clinical info (icu, gender, age) from the series matrix
patients, gsm_ids, short_ids = [], [], []
info = {}
with open(matrix_path) as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        label = parts[0]
        values = [v.strip('"') for v in parts[1:]]

        if label == "!Sample_title":
            patients = values
        elif label == "!Sample_geo_accession":
            gsm_ids = values
        elif label == "!Sample_description" and not short_ids:
            short_ids = values            # first description line: C1, C2, ... NC26
        elif label == "!Sample_characteristics_ch1":
            name = values[0].split(":")[0]
            if name in ("icu", "Sex", "age (years)"):
                info[name] = [v.split(":", 1)[1].strip() for v in values]

clinical = pd.DataFrame(info, index=patients).T
clinical = clinical.rename(index={"Sex": "gender", "age (years)": "age"})
clinical = clinical.loc[["icu", "gender", "age"]]

# 2. gene expression (TPM)
expr = pd.read_csv(tpm_path, sep="\t", index_col=0)   # first column = gene symbols
print("Expression table:", expr.shape)
print("Expression columns look like:", list(expr.columns[:3]))

# the columns of the expression file need to be renamed to the patient names.
# try each possible naming scheme and use the one that matches the most columns
candidates = {
    "short id (C1, NC1...)": dict(zip(short_ids, patients)),
    "GSM id": dict(zip(gsm_ids, patients)),
    "patient name": dict(zip(patients, patients)),
}
best = max(candidates, key=lambda k: sum(c in candidates[k] for c in expr.columns))
mapping = candidates[best]
print("Matching columns by:", best)

matched = [c for c in expr.columns if c in mapping]
if len(matched) < len(patients):
    print("WARNING: only", len(matched), "of", len(patients), "samples matched")

expr = expr[matched].rename(columns=mapping)
expr = expr[clinical.columns]            # same patient order as the clinical table

#3. put everything in one dataframe
df = pd.concat([clinical, expr])         # rows: icu, gender, age, then all genes
print(df.shape)
print(df.head(19472))

Expression table: (19472, 126)
Expression columns look like: ['C1', 'C2', 'C3']
Matching columns by: short id (C1, NC1...)
(19475, 126)
       COVID_01_39y_male_NonICU COVID_02_63y_male_NonICU  \
icu                          no                       no   
gender                     male                     male   
age                          39                       63   
A1BG                       0.49                     0.29   
A1CF                        0.0                      0.0   
...                         ...                      ...   
ZXDA                       0.95                      0.6   
ZXDB                       1.63                     1.15   
ZXDC                      15.51                    15.62   
ZYG11A                     0.06                     0.14   
ZYG11B                     8.17                      8.2   

       COVID_03_33y_male_NonICU COVID_04_49y_male_NonICU  \
icu                          no                       no   
gender                 